# nnU-Net v2 — WL Bruise Segmentation (ORC / SLURM Jupyter Lab)

Runs the **same** native nnU-Net pipeline as the GPU-server package (`train_nnunet_baseline.py`: *convert → plan → split → train → predict → score*).

### ORC session assumptions
- Launched via OnDemand Jupyter Lab with `--notebook-dir=/scratch/$USER`, **GPU** partition, GPU type **3g.40gb** (a 40 GB MIG slice ≈ 3/7 of an A100), 1 GPU, 8 cores, 12 h wall-time.
- Everything lives on **`/scratch/$USER`** — it persists between sessions, so checkpoints survive a 12 h wall-time kill and you can **resume** (no Google Drive needed).

### Before you run
1. Copy **`nnunet_bruise_pipeline.zip`** (the corrected 2.45 GB package) to `/scratch/$USER/` (e.g. `scp` / Globus / `cp` from your home dir).
2. Make sure this Jupyter kernel has **PyTorch with GPU** and **nnU-Net** available (cell 2 checks; cell 4 installs nnU-Net if missing). If `torch.cuda.is_available()` is False, switch to a conda kernel that has GPU torch — the launcher's CUDA/CUDNN module pickers only matter for a *system* torch build.

### Epoch budget vs the 12 h wall-time
- MIG 3g.40gb is ~40% of an A100 → epochs are slower than a full card. **Measure first** with the 1-epoch smoke test (set `EPOCHS = 1`), read the per-epoch seconds from nnU-Net's log, then size the run.
- **Full 1000 epochs** likely exceeds 12 h on this slice → run it across **multiple sessions**: submit again later, set `RESUME = True`, re-run cells — training continues from the last `/scratch` checkpoint.
- Default below is **250** (a clean single-session baseline).

## 1 · GPU / SLURM / env check

In [ ]:
import os
!nvidia-smi
import torch
print('\npython env :', os.sys.executable)
print('torch      :', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.is_available())
print('SLURM job  :', os.environ.get('SLURM_JOB_ID','?'),
      '| cpus', os.environ.get('SLURM_CPUS_PER_TASK', os.environ.get('SLURM_CPUS_ON_NODE','?')),
      '| gpu', os.environ.get('SLURM_JOB_GPUS', os.environ.get('CUDA_VISIBLE_DEVICES','?')))
assert torch.cuda.is_available(), 'GPU not visible to torch — switch to a CUDA-enabled kernel/conda env.'

## 2 · Config — edit these
Paths default to `/scratch/$USER`. `EPOCHS` accepts a preset trainer size (`1 5 10 20 50 100 250 500 750 2000 4000 8000`, or `1000` for the full default).

In [ ]:
import os
USER = os.environ.get('USER', os.environ.get('LOGNAME', 'user'))
SCRATCH = f'/scratch/{USER}'

# ===== EDIT ME =====
ZIP_PATH     = f'{SCRATCH}/nnunet_bruise_pipeline.zip'   # where you copied the 2.45 GB package
WORK         = f'{SCRATCH}/nnunet_bruise_pipeline'       # unzip target (persistent /scratch)

DATASET_ID   = 501
DATASET_NAME = 'WLBruise'
CONFIG       = '2d'
FOLD         = 0
EPOCHS       = 250      # 1 = smoke test. 1000 = full default. Presets: 1 5 10 20 50 100 250 500 750 ...
RESUME       = False    # True in a later session to continue from the /scratch checkpoint
print('USER =', USER, '| WORK =', WORK, '| EPOCHS =', EPOCHS, '| RESUME =', RESUME)

## 3 · Unzip the package (on `/scratch`, persistent)

In [ ]:
import os, zipfile, time
assert os.path.exists(ZIP_PATH), f'Zip not found at {ZIP_PATH} — copy nnunet_bruise_pipeline.zip to {SCRATCH}/ first.'
if not os.path.isdir(WORK):
    t = time.time()
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(SCRATCH)   # archive holds a top-level nnunet_bruise_pipeline/ folder
    print(f'unzipped in {time.time()-t:.0f}s')
else:
    print('already unzipped:', WORK)
assert os.path.isdir(f'{WORK}/data/train/images'), 'data/ missing — is this the full 2.45 GB zip (not the code-only patch)?'
# guard: ensure this is the CORRECTED script (epochs->trainer-variant fix), not a stale zip
_sig = open(f'{WORK}/train_nnunet_baseline.py').read()
assert '_resolve_trainer' in _sig, 'Stale zip: train_nnunet_baseline.py lacks the epochs fix. Copy the corrected zip.'
!ls {WORK}

## 4 · Ensure nnU-Net is installed
If the kernel already has `nnunetv2`, this is a no-op. Otherwise it tries `pip install --user` (works if the ORC compute node has internet/a proxy). If that fails, create a conda env with `torch`+`nnunetv2` on the **login node** first and relaunch Jupyter against it.

In [ ]:
import importlib, subprocess, sys
def _have(m):
    try: importlib.import_module(m); return True
    except Exception: return False
if _have('nnunetv2'):
    print('nnunetv2 already present — nothing to install.')
else:
    print('nnunetv2 missing; attempting pip install --user ...')
    rc = subprocess.call([sys.executable, '-m', 'pip', 'install', '--user', '-q',
                          'nnunetv2>=2.5', 'opencv-python-headless'])
    if rc != 0 or not _have('nnunetv2'):
        raise SystemExit('Install failed (likely no internet on the compute node). '
                         'Pre-build a conda env with torch+nnunetv2 on the login node and use that kernel.')
    print('installed nnunetv2 into ~/.local')
import nnunetv2, cv2; print('nnunetv2 ready | cv2', cv2.__version__)

## 5 · nnU-Net workspace env vars + ORC fixes
All three dirs live under `/scratch` (persistent → checkpoints survive a wall-time kill). Dataloader/preprocess worker counts are pinned to the **allocated** cores so the MIG slice doesn't over-subscribe the node's full CPU list. Two ORC-specific fixes are applied here:
- **PATH**: add `~/.local/bin` so the `nnUNetv2_*` console scripts (from `pip install --user`) are found.
- **LD_LIBRARY_PATH**: strip system cuda/cudnn dirs — the pip PyTorch wheel bundles its own cuDNN, and a mismatched system cuDNN on the path otherwise crashes training.

In [ ]:
import os, sys, shutil
os.environ['nnUNet_raw']          = f'{WORK}/nnunet/raw'
os.environ['nnUNet_preprocessed'] = f'{WORK}/nnunet/preprocessed'
os.environ['nnUNet_results']      = f'{WORK}/nnunet/results'
for k in ('nnUNet_raw', 'nnUNet_preprocessed', 'nnUNet_results'):
    os.makedirs(os.environ[k], exist_ok=True); print(k, '=', os.environ[k])

# (a) console scripts on PATH
for b in [os.path.expanduser('~/.local/bin'), os.path.dirname(sys.executable)]:
    if os.path.isdir(b) and b not in os.environ['PATH'].split(os.pathsep):
        os.environ['PATH'] = b + os.pathsep + os.environ['PATH']
print('nnUNetv2_train ->', shutil.which('nnUNetv2_train'))

# (b) let PyTorch use its bundled cuDNN (drop system cuda/cudnn from LD_LIBRARY_PATH)
kept = [p for p in os.environ.get('LD_LIBRARY_PATH','').split(':')
        if p and 'cuda' not in p.lower() and 'cudnn' not in p.lower()]
os.environ['LD_LIBRARY_PATH'] = ':'.join(kept)
import torch
print('cuDNN:', torch.backends.cudnn.version(), '| gpu:', torch.cuda.is_available())

# (c) dataloader/preprocess workers pinned to allocation; skip flaky torch.compile on MIG
ncpu = int(os.environ.get('SLURM_CPUS_PER_TASK', os.environ.get('SLURM_CPUS_ON_NODE', '8')))
nproc = max(2, ncpu - 1)
os.environ['nnUNet_n_proc_DA'] = str(nproc)
os.environ['nnUNet_def_n_proc'] = str(nproc)
os.environ['nnUNet_compile']    = 'f'
print(f'cores={ncpu} -> DA/preproc workers={nproc}')

## 6 · Verify the data (697/134 split, files present)

In [ ]:
%cd {WORK}
!python check_package.py

## 7 · Prep: convert → plan → split
Writes nnU-Net raw (3-channel RGB PNGs, 0/1 masks), fingerprints + preprocesses, and writes `splits_final.json` fold 0 from the **canonical 697/134 split**. (Idempotent — safe to re-run on resume.)

In [ ]:
!python -u train_nnunet_baseline.py \
  --train-manifest manifests/train_manifest.csv \
  --test-manifest  manifests/test_manifest.csv \
  --data-root "{WORK}" --out-dir "{WORK}/results_nnunet" \
  --dataset-id {DATASET_ID} --dataset-name {DATASET_NAME} \
  --configuration {CONFIG} --fold {FOLD} \
  --convert --plan --split

## 8 · Train (single fold, native pipeline)
Live epoch logs stream below (and to `nnUNet_results/.../fold_0/training_log_*.txt`). If the 12 h wall-time kills the job mid-run, just relaunch, set `RESUME = True`, re-run cells 1–7, then re-run this cell — nnU-Net continues from the last checkpoint on `/scratch`.

In [ ]:
resume_flag = '--continue-train' if RESUME else ''
epoch_flag  = f'--epochs {EPOCHS}' if int(EPOCHS) != 1000 else ''  # 1000 = nnU-Net default (omit flag)
cmd = f'''python -u train_nnunet_baseline.py \
  --train-manifest manifests/train_manifest.csv \
  --test-manifest  manifests/test_manifest.csv \
  --data-root "{WORK}" --out-dir "{WORK}/results_nnunet" \
  --dataset-id {DATASET_ID} --dataset-name {DATASET_NAME} \
  --configuration {CONFIG} --fold {FOLD} \
  {epoch_flag} {resume_flag} \
  --train'''
print(cmd, '\n' + '='*70)
get_ipython().system(cmd)

## 9 · Predict + score
Inference on the 185 consensus test images + Dice / IoU / complete-miss at matched 640 geometry (identical metric code to the server + SMP baselines).

In [ ]:
epoch_flag = f'--epochs {EPOCHS}' if int(EPOCHS) != 1000 else ''  # must match training so predict loads the same trainer
cmd = f'''python -u train_nnunet_baseline.py \
  --train-manifest manifests/train_manifest.csv \
  --test-manifest  manifests/test_manifest.csv \
  --data-root "{WORK}" --out-dir "{WORK}/results_nnunet" \
  --dataset-id {DATASET_ID} --dataset-name {DATASET_NAME} \
  --configuration {CONFIG} --fold {FOLD} \
  {epoch_flag} \
  --predict --score'''
get_ipython().system(cmd)

## 10 · Results
Everything is already on persistent `/scratch` — nothing to copy back.

In [ ]:
import json
res = f'{WORK}/results_nnunet/results/nnunet_FINAL.json'
summ = json.load(open(res))
print(json.dumps(summ, indent=2))
print('\n  mean Dice   %.4f' % summ['mean_dice'])
print('  median Dice %.4f'   % summ['median_dice'])
print('  mean IoU    %.4f'   % summ['mean_iou'])
print('  miss rate   %.2f%%  (%d/%d)' % (summ['complete_miss_rate']*100, summ['complete_miss_count'], summ['n_images']))
print('\n  full outputs :', f'{WORK}/results_nnunet/results')
print('  predictions  :', f'{WORK}/results_nnunet/nnunet_test_pred')